# 02. Garbage / Litter: YOLOv8 segmentation on TACO

Converts TACO (COCO polygons, 60 litter categories) into a single-class YOLO segmentation dataset called `Litter`, then trains `garbage_litter_model.pt`.

Steps: match each image's orientation to its annotation size (some photos carry EXIF rotation), resize so the long side is at most 1,280 px, write normalised polygons, and split 85:15 with seed 0.

Our run: 1,428 images (1,214 train, 214 val), 45 epochs, about 52 minutes. Validation mask mAP50 0.502, precision 0.757, recall 0.434. Run `00_download_datasets.ipynb` first.

In [ ]:
from pathlib import Path

# Works whether the notebook is opened from ai_model/ or ai_model/notebooks/.
ROOT = Path.cwd().resolve()
while ROOT.name != "ai_model" and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert ROOT.name == "ai_model", "Open this notebook from inside the ai_model folder"

DATA = ROOT / "data"          # downloaded and converted datasets (git-ignored)
RAW = DATA / "raw"            # original downloads
RUNS = ROOT / "runs"          # Ultralytics training runs (git-ignored)
WEIGHTS = ROOT / "weights"    # final model files used by the AI service
RESULTS = ROOT / "results"    # metrics used in the project report
for p in (DATA, RAW, RUNS, WEIGHTS, RESULTS):
    p.mkdir(parents=True, exist_ok=True)
print("ai_model root:", ROOT)

In [ ]:
import torch

# Apple Silicon uses the MPS backend; an NVIDIA GPU uses CUDA; otherwise CPU.
DEVICE = 0 if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("training device:", DEVICE)

In [ ]:
import json, random
from PIL import Image, ImageOps

NAME = "garbage_litter"
taco = RAW / "taco"
out = DATA / NAME
data_yaml = out / "data.yaml"

if not (out / "images" / "train").exists():
    coco = json.load(open(taco / "annotations.json"))
    anns = {}
    for a in coco["annotations"]:
        anns.setdefault(a["image_id"], []).append(a)
    images = [im for im in coco["images"] if (taco / "images" / im["file_name"].replace("/", "_")).exists()]
    random.seed(0)
    random.shuffle(images)
    n_val = round(len(images) * 0.15)
    counts = {"train": 0, "val": 0}
    for i, im in enumerate(images):
        split = "val" if i < n_val else "train"
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)
        raw = Image.open(taco / "images" / im["file_name"].replace("/", "_"))
        raw.load()
        W, H = im["width"], im["height"]
        # keep whichever orientation matches the annotation's aspect ratio
        pic = min([raw, ImageOps.exif_transpose(raw)], key=lambda p: abs(p.size[0] / p.size[1] - W / H)).convert("RGB")
        s = 1280 / max(pic.size)
        if s < 1:
            pic = pic.resize((round(pic.size[0] * s), round(pic.size[1] * s)), Image.LANCZOS)
        stem = f"taco_{im['id']:04d}"
        pic.save(out / "images" / split / f"{stem}.jpg", quality=90)
        lines = []
        for a in anns.get(im["id"], []):
            for poly in a["segmentation"] if isinstance(a["segmentation"], list) else []:
                if len(poly) < 6:
                    continue
                pts = " ".join(f"{min(max(poly[k] / W, 0), 1):.5f} {min(max(poly[k + 1] / H, 0), 1):.5f}"
                               for k in range(0, len(poly) - 1, 2))
                lines.append("0 " + pts)
        (out / "labels" / split / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
        counts[split] += 1
    json.dump({"n_total": len(coco["images"]), "n_used": len(images), "n_train": counts["train"], "n_val": counts["val"]},
              open(RESULTS / "garbage_split.json", "w"))
    print(counts)

data_yaml.write_text(f"path: {out}\ntrain: images/train\nval: images/val\nnames: ['Litter']\n")
for split in ("train", "val"):
    print(split, len(list((out / "images" / split).glob("*.jpg"))), "images")

In [ ]:
import shutil
from ultralytics import YOLO


def train(name, data_yaml, task, epochs):
    """Fine-tune a COCO-pretrained YOLOv8 nano model with the settings used for every UrbanFix category."""
    base = "yolov8n-seg.pt" if task == "segment" else "yolov8n.pt"
    model = YOLO(base)
    model.train(
        data=str(data_yaml),
        epochs=epochs,
        imgsz=640,
        batch=16,
        patience=15,          # stop if validation fitness does not improve for 15 epochs
        device=DEVICE,
        seed=0,
        deterministic=False,
        workers=4,
        plots=True,
        project=str(RUNS),
        name=name,
        exist_ok=True,
        verbose=False,
    )
    best = RUNS / name / "weights" / "best.pt"
    target = WEIGHTS / f"{name}_model.pt"
    shutil.copy(best, target)
    print("saved", target)
    return target

In [ ]:
train(NAME, data_yaml, task="segment", epochs=45)

In [ ]:
def quick_val(name, data_yaml):
    """Validate the saved model and print the headline numbers."""
    m = YOLO(str(WEIGHTS / f"{name}_model.pt"))
    r = m.val(data=str(data_yaml), imgsz=640, batch=16, device=DEVICE, plots=False, verbose=False,
              project=str(RUNS / "val"), name=name, exist_ok=True)
    print(f"box  P {r.box.mp:.3f}  R {r.box.mr:.3f}  mAP50 {r.box.map50:.3f}  mAP50-95 {r.box.map:.3f}")
    if hasattr(r, "seg") and r.seg is not None and m.task == "segment":
        print(f"mask P {r.seg.mp:.3f}  R {r.seg.mr:.3f}  mAP50 {r.seg.map50:.3f}  mAP50-95 {r.seg.map:.3f}")
    return r

In [ ]:
quick_val(NAME, data_yaml);